# Chapter 4 - Representing Data and Engineering Features

**Book:** Introduction to Machine Learning with Python  
**Authors:** Andreas C. Müller & Sarah Guido

Chapter ini membahas bagaimana merepresentasikan data agar dapat
digunakan oleh machine learning serta bagaimana membuat dan memilih
fitur yang sesuai.

## 1. Chapter Overview

Tidak semua data berbentuk angka. Dataset machine learning dapat
memiliki data numerik maupun kategorikal.

Chapter ini membahas:

1. Categorical Variables
2. One-Hot Encoding
3. ColumnTransformer
4. Binning / Discretization
5. Polynomial Features
6. Nonlinear Transformations
7. Feature Selection
8. Expert Knowledge

In [3]:
import pandas as pd

data = pd.DataFrame({
    "city": [
        "Bandung", "Jakarta", "Bandung", "Surabaya",
        "Jakarta", "Bandung", "Surabaya", "Jakarta",
        "Bandung", "Surabaya", "Jakarta", "Bandung"
    ],
    "age": [
        20, 25, 22, 30,
        27, 21, 35, 29,
        24, 32, 26, 23
    ],
    "income": [
        4, 7, 5, 8,
        6, 5, 9, 7,
        5, 8, 6, 4
    ],
    "purchased": [
        0, 1, 0, 1,
        1, 0, 1, 1,
        0, 1, 1, 0
    ]
})

data

,city,age,income,purchased
0,Bandung,20,4,0
1,Jakarta,25,7,1
2,Bandung,22,5,0
3,Surabaya,30,8,1
4,Jakarta,27,6,1
5,Bandung,21,5,0
6,Surabaya,35,9,1
7,Jakarta,29,7,1
8,Bandung,24,5,0
9,Surabaya,32,8,1


## 3. Categorical Variables

Categorical variables adalah fitur yang berisi kategori.

Contoh:
- City
- Gender
- Jenis produk
- Jenis pekerjaan

Machine learning umumnya membutuhkan representasi numerik,
sehingga categorical variables perlu diubah terlebih dahulu.

## 4. One-Hot Encoding

One-hot encoding mengubah setiap kategori menjadi kolom binary.

Contoh:

Bandung → [1, 0, 0]
Jakarta  → [0, 1, 0]
Surabaya → [0, 0, 1]

Dengan cara ini, model tidak menganggap bahwa suatu kategori
memiliki nilai yang lebih tinggi atau lebih rendah dari kategori lain.

## 6. make_column_transformer

`make_column_transformer` merupakan cara yang lebih singkat
untuk membuat ColumnTransformer.

Tujuannya sama, yaitu menerapkan preprocessing yang berbeda
pada kolom yang berbeda.

In [4]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

city_encoded = encoder.fit_transform(
    data[["city"]]
)

print(city_encoded)
print("Nama fitur:", encoder.get_feature_names_out(["city"]))

[[1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
Nama fitur: ['city_Bandung' 'city_Jakarta' 'city_Surabaya']


## 5. ColumnTransformer

Dalam dataset nyata, kita dapat memiliki beberapa tipe fitur sekaligus.

Contohnya:

- city → kategorikal
- age → numerik
- income → numerik

ColumnTransformer memungkinkan kita memberikan preprocessing
yang berbeda untuk setiap jenis kolom.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# Pisahkan fitur dan target
X = data[["city", "age", "income"]]
y = data["purchased"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "city",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["city"]
        )
    ],
    remainder="passthrough"
)

# Fit hanya pada data training
X_train_transformed = preprocessor.fit_transform(X_train)

# Transform data testing
X_test_transformed = preprocessor.transform(X_test)

print("Training shape:", X_train_transformed.shape)
print("Testing shape :", X_test_transformed.shape)

Training shape: (9, 5)
Testing shape : (3, 5)


## 6. Applying Machine Learning Model

Setelah data kategorikal diubah menjadi numerik,
hasil preprocessing dapat digunakan oleh machine learning model.

Pada contoh ini digunakan Logistic Regression.

In [6]:
model = LogisticRegression(
    max_iter=1000
)

model.fit(
    X_train_transformed,
    y_train
)

accuracy = model.score(
    X_test_transformed,
    y_test
)

print("Accuracy:", accuracy)

Accuracy: 1.0


## 7. Binning and Discretization

Binning mengubah data numerik menjadi beberapa kelompok atau interval.

Contohnya:

Age 18–25 → Young  
Age 26–40 → Adult  
Age 41+   → Senior

Teknik ini dapat membantu model tertentu dalam menangkap pola
yang tidak linear.

In [7]:
from sklearn.preprocessing import KBinsDiscretizer

age = data[["age"]]

binning = KBinsDiscretizer(
    n_bins=3,
    encode="ordinal",
    strategy="uniform"
)

age_binned = binning.fit_transform(age)

print(age_binned[:10])

[[0.]
 [1.]
 [0.]
 [2.]
 [1.]
 [0.]
 [2.]
 [1.]
 [0.]
 [2.]]


## 8. Polynomial Features

Polynomial Features digunakan untuk membuat fitur baru dari
kombinasi fitur yang sudah ada.

Contohnya:

x₁
x₂
x₁²
x₂²
x₁ × x₂

Hal ini memungkinkan model linear mempelajari hubungan yang lebih kompleks.

In [8]:
from sklearn.preprocessing import PolynomialFeatures

X_simple = data[["age", "income"]]

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

X_poly = poly.fit_transform(X_simple)

print("Shape awal :", X_simple.shape)
print("Shape baru :", X_poly.shape)

print(
    poly.get_feature_names_out(
        ["age", "income"]
    )
)

Shape awal : (12, 2)
Shape baru : (12, 5)
['age' 'income' 'age^2' 'age income' 'income^2']


## 9. Nonlinear Transformations

Transformasi nonlinear dapat digunakan ketika hubungan antara
fitur dan target tidak bersifat linear.

Contoh transformasi:

- Logarithmic
- Square root
- Polynomial

Transformasi harus disesuaikan dengan karakteristik data.

In [9]:
import numpy as np

income = data["income"].values

log_income = np.log(income)

result = pd.DataFrame({
    "income": income,
    "log_income": log_income
})

result.head()

,income,log_income
0,4,1.386294
1,7,1.945910
2,5,1.609438
3,8,2.079442
4,6,1.791759


## 10. Feature Selection

Tidak semua fitur memberikan informasi yang berguna.

Feature selection bertujuan memilih fitur yang paling relevan.

Manfaat:

- Mengurangi kompleksitas model.
- Mengurangi noise.
- Mempercepat proses training.
- Membuat model lebih sederhana.

Beberapa pendekatan:

1. Univariate Statistics
2. Model-Based Selection
3. Iterative Selection

In [10]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, f_classif

cancer = load_breast_cancer()

X_cancer = cancer.data
y_cancer = cancer.target

print("Jumlah data  :", X_cancer.shape[0])
print("Jumlah fitur :", X_cancer.shape[1])

Jumlah data  : 569
Jumlah fitur : 30


In [11]:
selector = SelectKBest(
    score_func=f_classif,
    k=10
)

X_selected = selector.fit_transform(
    X_cancer,
    y_cancer
)

print("Fitur awal    :", X_cancer.shape[1])
print("Fitur terpilih:", X_selected.shape[1])

Fitur awal    : 30
Fitur terpilih: 10


SelectKBest memilih sejumlah fitur berdasarkan skor statistik.

Pada contoh ini, hanya 10 fitur dengan skor terbaik yang dipertahankan.

## 12. Model-Based Feature Selection

Model-based selection menggunakan machine learning model
untuk menentukan fitur yang penting.

Random Forest dapat memberikan nilai feature importance
yang kemudian digunakan untuk memilih fitur.

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

forest.fit(
    X_cancer,
    y_cancer
)

selector_model = SelectFromModel(
    forest,
    threshold="median"
)

X_selected_model = selector_model.transform(
    X_cancer
)

print("Fitur awal    :", X_cancer.shape[1])
print("Fitur terpilih:", X_selected_model.shape[1])

Fitur awal    : 30
Fitur terpilih: 15


## 13. Recursive Feature Elimination (RFE)

RFE melakukan feature selection secara bertahap.

Model akan:

1. Melatih model.
2. Menilai kepentingan fitur.
3. Menghapus fitur yang paling tidak penting.
4. Mengulangi proses sampai jumlah fitur yang diinginkan tercapai.

In [13]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=5000
)

rfe = RFE(
    estimator=logreg,
    n_features_to_select=10
)

rfe.fit(
    X_cancer,
    y_cancer
)

print("Jumlah fitur terpilih:", rfe.n_features_)

Jumlah fitur terpilih: 10


## 14. Utilizing Expert Knowledge

Feature engineering juga dapat dilakukan menggunakan pengetahuan
dari bidang tertentu.

Contoh data transaksi:

price × quantity
↓
total_revenue

Contoh data waktu:

timestamp
↓
hour
day
month
is_weekend

Dengan memahami domain permasalahan, kita dapat membuat fitur
yang lebih bermakna bagi model.

In [14]:
data["total_income"] = data["income"] * data["age"]

data[[
    "age",
    "income",
    "total_income"
]].head()

,age,income,total_income
0,20,4,80
1,25,7,175
2,22,5,110
3,30,8,240
4,27,6,162


## 15. Comparison of Feature Engineering Methods

| Metode | Fungsi |
|---|---|
| One-Hot Encoding | Mengubah kategori menjadi angka |
| ColumnTransformer | Memberikan preprocessing berbeda pada kolom |
| Binning | Membagi data numerik menjadi interval |
| Polynomial Features | Membuat fitur/interaksi baru |
| Nonlinear Transformation | Mengubah bentuk data |
| Univariate Selection | Memilih fitur berdasarkan statistik |
| Model-Based Selection | Memilih berdasarkan model |
| RFE | Memilih fitur secara iteratif |
| Expert Knowledge | Membuat fitur berdasarkan domain |

## 16. Results and Analysis

Eksperimen menunjukkan bahwa representasi data merupakan bagian
penting dalam machine learning.

Categorical variables perlu diubah menjadi bentuk numerik,
misalnya menggunakan one-hot encoding.

ColumnTransformer memungkinkan preprocessing berbeda untuk
tipe fitur yang berbeda.

Feature engineering dapat membuat fitur baru yang lebih informatif,
sedangkan feature selection membantu mengurangi fitur yang kurang
relevan.

Pemilihan metode harus disesuaikan dengan karakteristik dataset
dan tujuan machine learning.

## 17. Chapter Summary

Chapter ini membahas representasi data dan feature engineering.

Hal-hal penting:

- Data kategorikal dapat diubah menggunakan one-hot encoding.
- ColumnTransformer digunakan untuk preprocessing beberapa jenis fitur.
- Binning mengubah fitur kontinu menjadi beberapa kelompok.
- Polynomial features membuat kombinasi dan interaksi fitur.
- Transformasi nonlinear dapat membantu menangani hubungan yang tidak linear.
- Feature selection dapat mengurangi fitur yang kurang relevan.
- Expert knowledge dapat digunakan untuk membuat fitur yang lebih bermakna.

## Key Takeaways

1. Representasi data sangat memengaruhi machine learning.
2. Data kategorikal perlu diproses dengan tepat.
3. Feature engineering dapat menghasilkan informasi baru.
4. Feature selection dapat menyederhanakan model.
5. Pengetahuan domain dapat membantu menghasilkan fitur yang lebih relevan.